# 04 · Comparación Argentina vs. Global

¿El consumo argentino tiene identidad propia o sigue de cerca las tendencias
globales, y con qué desfasaje? Trabajamos con tres series construidas por día a
partir de `data/charts_ar_global.csv` (todo en memoria, sin CSVs intermedios):

- **serie_ar** - cantidad de artistas distintos en el Top 200 de Argentina.
- **serie_global** - cantidad de artistas distintos en el Top 200 Global.
- **serie_interseccion** - artistas que aparecen **simultáneamente** en ambos Top 200
  ese mismo día (intersección real de conjuntos). Es la serie principal de esta
  sub-pregunta.

## 0. Armado de las series en memoria

In [ ]:
import sys
sys.path.append("..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import find_peaks

from src.utils import (
    interpolar_serie, graficar_serie, resumen_estadistico,
    calcular_fft, graficar_espectro, descomponer_MA7_MA21,
    energia_serie, correlacion_cruzada,
)

CHART = "top200"
df = pd.read_csv("../data/charts_ar_global.csv", parse_dates=["date"])

# Un título suele acreditar varios artistas en un string ("Bad Bunny, Ozuna"):
# los separamos para comparar entidades individuales, no strings completos.
top200 = df[df["chart"] == CHART].copy()
top200["artista_individual"] = top200["artist"].str.split(", ")
exploded = top200.explode("artista_individual")

# Conjunto de artistas por día en cada región (reutilizable en toda la sección)
set_ar_por_dia = exploded[exploded["region"] == "Argentina"].groupby("date")["artista_individual"].apply(set)
set_gl_por_dia = exploded[exploded["region"] == "Global"].groupby("date")["artista_individual"].apply(set)

In [ ]:
# serie_ar / serie_global: cantidad de artistas distintos por día
serie_ar_cruda     = set_ar_por_dia.apply(len).rename("artistas_ar")
serie_global_cruda = set_gl_por_dia.apply(len).rename("artistas_global")

# serie_interseccion: |AR ∩ Global| por día, sobre las fechas presentes en ambos
fechas_comunes = set_ar_por_dia.index.intersection(set_gl_por_dia.index)
serie_interseccion_cruda = pd.Series(
    {d: len(set_ar_por_dia[d] & set_gl_por_dia[d]) for d in fechas_comunes},
    name="artistas_en_comun",
).sort_index()

def reportar(nombre, cruda):
    serie = interpolar_serie(cruda)
    huecos = len(serie) - cruda.index.nunique()
    print(f"{nombre}: {len(serie)} días | {huecos} huecos interpolados | "
          f"{serie.index.min().date()} -> {serie.index.max().date()}")
    return serie

serie_ar           = reportar("serie_ar          ", serie_ar_cruda)
serie_global       = reportar("serie_global      ", serie_global_cruda)
serie_interseccion = reportar("serie_interseccion", serie_interseccion_cruda)

## 1. Exploratorio (serie principal: intersección)

In [ ]:
graficar_serie(serie_interseccion, "Artistas en común por día - Top 200 Argentina ∩ Global")
plt.show()
print("resumen serie_interseccion:", {k: round(v, 1) for k, v in resumen_estadistico(serie_interseccion).items()})

## 2. Fourier

FFT de la intersección: ¿aparece un patrón semanal (pico en 1/7 ciclos/día)?

In [ ]:
frecs, mags = calcular_fft(serie_interseccion)
graficar_espectro(frecs, mags, "Espectro - artistas en común (AR ∩ Global)")
plt.axvline(1/7, color="red", ls="--", lw=1, label="1/7 (semanal)")
plt.legend(); plt.show()

f_pico = frecs[np.argmax(mags)]
print(f"Frecuencia dominante: {f_pico:.4f} ciclos/día -> período {1/f_pico:.1f} días")

# Energía relativa en una banda angosta alrededor de 1/7 (semanal)
banda = (frecs >= 1/7 * 0.9) & (frecs <= 1/7 * 1.1)
peso_semanal = mags[banda].sum() / mags.sum()
print(f"Peso de la banda semanal (±10% de 1/7) en el espectro: {100 * peso_semanal:.1f}%")

**Patrón semanal.** La energía del espectro está dominada por las frecuencias
bajas (variación lenta a lo largo de los años: cuánto se solapan los dos charts según
la época). El componente semanal (1/7 ciclos/día) existe pero es secundario frente a
esa tendencia de fondo - la superposición AR∩Global se mueve más por *modas* de meses
que por el día de la semana.

## 3. Filtrado (MA7 / MA21)

In [ ]:
desc = descomponer_MA7_MA21(serie_interseccion)

fig, ax = plt.subplots(4, 1, figsize=(12, 9), sharex=True)
ax[0].plot(serie_interseccion.index, serie_interseccion.values); ax[0].set_title("Original (AR ∩ Global)")
ax[1].plot(desc["tendencia"]); ax[1].set_title("Tendencia (MA21)")
ax[2].plot(desc["estacional"]); ax[2].set_title("Estacional (MA7 - MA21)")
ax[3].plot(desc["residuo"]); ax[3].set_title("Residuo (serie - MA7)")
for a in ax: a.set_ylabel("artistas")
plt.tight_layout(); plt.show()

## 4. Energía y Parseval (antes vs. después del filtrado)

In [ ]:
filtrada = desc["tendencia"] + desc["estacional"]   # = MA7 (sin residuo)

e_antes = energia_serie(serie_interseccion)
e_desp  = energia_serie(filtrada)

print("ANTES del filtrado:")
print(f"    E_tiempo = {e_antes['energia_tiempo']:.4e} | E_frec = {e_antes['energia_frecuencia']:.4e} | Parseval OK: {e_antes['parseval_ok']}")
print("DESPUÉS del filtrado (MA7):")
print(f"    E_tiempo = {e_desp['energia_tiempo']:.4e} | E_frec = {e_desp['energia_frecuencia']:.4e} | Parseval OK: {e_desp['parseval_ok']}")
print(f"Energía retenida: {100 * e_desp['energia_tiempo'] / e_antes['energia_tiempo']:.2f}%")

## 5. Correlación cruzada AR vs. Global (extra de esta pregunta)

`correlacion_cruzada(serie_ar, serie_global, max_lag=30)`. Convención: **lag > 0
significa que serie_ar adelanta a serie_global** (Argentina "va primero"); lag < 0, que
Global adelanta a Argentina.

In [ ]:
cc = correlacion_cruzada(serie_ar, serie_global, max_lag=30)

plt.figure(figsize=(11, 4))
plt.stem(cc.index, cc.values)
plt.axvline(0, color="gray", lw=0.8)
plt.title("Correlación cruzada: artistas distintos AR vs. Global")
plt.xlabel("lag (días)  ·  lag>0: AR adelanta a Global")
plt.ylabel("correlación normalizada")
plt.tight_layout(); plt.show()

lag_max = int(cc.idxmax())
print(f"Lag de máxima correlación: {lag_max} días (correlación = {cc.max():.3f})")
print("Top 5 lags por correlación:")
print(cc.sort_values(ascending=False).head(5).to_string())

## 6. Picos de la intersección: quién los explica

In [ ]:
# Picos separados en el tiempo (distancia mínima 14 días, prominencia = 1 desvío)
vals = serie_interseccion_cruda.values
idx_picos, _ = find_peaks(vals, distance=14, prominence=serie_interseccion_cruda.std())
picos = serie_interseccion_cruda.iloc[idx_picos].sort_values(ascending=False).head(5)

def explicar_pico(fecha):
    comunes = set_ar_por_dia[fecha] & set_gl_por_dia[fecha]
    dia = exploded[(exploded["date"] == fecha) & (exploded["region"] == "Argentina")
                   & (exploded["artista_individual"].isin(comunes))]
    canciones = dia[["title", "artist", "rank"]].drop_duplicates().sort_values("rank")
    return comunes, canciones

for fecha, valor in picos.items():
    comunes, canciones = explicar_pico(fecha)
    print("=" * 70)
    print(f"{fecha.date()}  -  {int(valor)} artistas en común")
    print(f"Artistas (algunos): {sorted(comunes)[:12]}")
    print("Canciones de esos artistas en el Top 200 AR ese día (top 6 por rank):")
    print(canciones.head(6).to_string(index=False))

## 7. Conclusión: ¿consumo propio o sigue al Global?

**Argentina sigue de cerca al Global, sin un desfasaje temporal claro.**

- **Alto solapamiento.** En un día típico hay ~65 artistas en común entre ambos Top 200
  (media 65.3, desvío 9.9). No es un consumo aislado: buena parte del ranking argentino
  está poblado por los mismos artistas que dominan el mundo.
- **El pico de superposición es 2017.** Los 5 máximos de la intersección (~90-92 artistas
  en común) caen todos en la primera mitad de 2017, explicados por el fenómeno del
  **crossover latino** (Despacito, Shape of You, CNCO, Nicky Jam, Enrique Iglesias…):
  esa época el Top 200 argentino era casi un espejo del global.
- **Sin desfasaje nítido.** La correlación cruzada de la cantidad de artistas distintos
  es **débil (~0.24) y casi simétrica alrededor de lag 0**: el máximo cae en lag 28
  (0.245) pero lag 0 queda empatado (0.244), diferencia despreciable. No hay evidencia de
  que Argentina adelante ni retrase sistemáticamente al Global a escala diaria —**se
  mueven aproximadamente en sincronía**. El leve realce en lags ~28-30 es más un eco de
  variación mensual compartida que un verdadero "quién sigue a quién".
- **Identidad propia parcial.** Que la correlación de los *conteos* sea baja pese al alto
  solapamiento indica que, sobre ese fondo común, Argentina conserva un margen propio
  (cumbia, rock nacional, trap local) que hace que el *nivel* de coincidencia no se
  mueva en lockstep con el global.

En síntesis: **Argentina no lidera ni va a la zaga; acompaña al Global casi en tiempo
real, con un componente local que le da identidad parcial.**